pipeline name:  PostgreSQL_bronze

| Approach             | Purpose                   |
| -------------------- | ------------------------- |
| `ingestion_metadata` | Track pipeline execution  |
| `updated_at`         | Track row changes         |
| `start_at/end_at`    | Track historical versions |


In [0]:
%sql
-- ingestion_metadata table is mainly used for incremental loading / watermark tracking in ETL or ELT pipelines.
CREATE TABLE IF NOT EXISTS healthcare0209.healthcare0209_bronze.ingestion_metadata
(
    table_name STRING,
    last_load_time TIMESTAMP
);

In [0]:
# PostgreSQL connection details
jdbc_url = "jdbc:postgresql://ep-damp-breeze-aprlavv3.c-7.us-east-1.aws.neon.tech:5432/neondb?sslmode=require"

user = "neondb_owner" 
password = "npg_8LkWQpc6faNM"
print("PostgreSQL connection established")

.option() is used to pass configuration/settings parameters to a reader, writer, or connector.
.option("key", "value")
"key" → configuration name
"value" → configuration value

Using Apache Spark (PySpark) to connect to a PostgreSQL database through JDBC and fetch all table names from the public schema.

--> "Read data using JDBC connection." JDBC = Java Database Connectivity
--> "Database connection URL. ".option("url", jdbc_url)
--> "Structure:" jdbc:postgresql://host:port/database

**Architecture Flow**
Spark
   ↓ JDBC
PostgreSQL
   ↓
information_schema.tables
   ↓
Get table names
   ↓
Store in Python list
   ↓
Loop and ingest tables

Instead of reading a table directly, Spark executes this SQL query inside PostgreSQL.


In [0]:
# Read the all tables
tables_df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option(
        "query",
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema='public'
        """
    ) \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

tables = [r.table_name for r in tables_df.collect()]
print(tables)

In [0]:
# ==========================================
# Get Primary Keys
# ==========================================

pk_df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option(
        "query",
        """
        SELECT
            tc.table_name,
            kcu.column_name
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
        WHERE tc.constraint_type='PRIMARY KEY'
        """
    ) \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .load()
display(pk_df)

In [0]:
pk_dict = {
    row["table_name"]: row["column_name"]
    for row in pk_df.collect()
}

print(pk_dict)

Source DB
   ↓
Read only changed records
   ↓
MERGE into Delta Bronze table
   ↓
Update ingestion metadata

**High Level Architecture**
PostgreSQL / SQL Server
        ↓
updated_at filter
        ↓
Incremental Records
        ↓
Spark DataFrame
        ↓
MERGE INTO Delta Table
        ↓
Update ingestion_metadata



In [0]:
# ==========================================
# Incremental Load + MERGE
# ==========================================
from delta.tables import DeltaTable
for table_name in tables:

    print("=" * 50)
    print(f"Processing Table : {table_name}")

    # --------------------------------------
    # Get Last Load Timestamp
    # --------------------------------------

    result = spark.sql(f"""
        SELECT MAX(last_load_time)
        FROM healthcare0209.healthcare0209_bronze.ingestion_metadata
        WHERE table_name = '{table_name}'
    """).collect()

    # Extract Timestamp -->[Row(max(last_load_time)=datetime)] -->result[0] → first row -->[0] → first column
    last_load_time = result[0][0]

    # First Load Handling
    if last_load_time is None:
        last_load_time = "1900-01-01 00:00:00"

    print(f"Last Load Time : {last_load_time}")

    # Read incremental records from the source table using the last load timestamp, Only fetch the records updated after the last load timestamp or inserted after the last load timestamp
    query = f"""
    (
        SELECT *
        FROM {table_name}
        WHERE updated_at > '{last_load_time}'
    ) src
    """

    # Reads changed rows from source DB.
    incremental_df = spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", query) \
        .option("user", user) \
        .option("password", password) \
        .load()

    if incremental_df.count() == 0:
        print(f"No new records for {table_name}")
        continue

    target_table = f"healthcare0209.healthcare0209_bronze.{table_name}"

    pk = pk_dict[table_name]

    delta_table = DeltaTable.forName(
        spark,
        target_table
    )

    # MERGE
    temp_view = f"{table_name}_source"

    incremental_df.createOrReplaceTempView(temp_view)

    cols = incremental_df.columns

    update_clause = ", ".join(
        [f"{c} = s.{c}" for c in cols if c != pk]
    )

    insert_cols = ", ".join(cols)

    insert_vals = ", ".join(
        [f"s.{c}" for c in cols]
    )

    # Condition	Action
    # Record exists	UPDATE
    # Record not exists	INSERT
    merge_sql = f"""
    MERGE INTO {target_table} t
    USING {temp_view} s
    ON t.{pk} = s.{pk}
    WHEN MATCHED THEN
    UPDATE SET {update_clause}
    WHEN NOT MATCHED THEN
    INSERT ({insert_cols})
    VALUES ({insert_vals})
    """

    print("========== MERGE SQL ==========")
    print(merge_sql)

    spark.sql(merge_sql)

    # -------------------------
    # STEP 7 GOES HERE
    # -------------------------

    max_updated_at = incremental_df.agg(
        {"updated_at": "max"}
    ).collect()[0][0]

    spark.sql(f"""
        DELETE FROM healthcare0209.healthcare0209_bronze.ingestion_metadata
        WHERE table_name = '{table_name}'
    """)

    spark.sql(f"""
        INSERT INTO healthcare0209.healthcare0209_bronze.ingestion_metadata
        VALUES (
            '{table_name}',
            TIMESTAMP('{max_updated_at}')
        )
    """)


In [0]:
%sql

select count(*) from healthcare0209.healthcare0209_bronze.billing

In [0]:
%sql
select * from healthcare0209.healthcare0209_bronze.ingestion_metadata